# Batch Normalization

## Learning Objectives
1. Implement the batch-norm forward pass from scratch with running statistics.
2. Compare BatchNorm, LayerNorm, and no-normalisation on a regression task.
3. Show how BN stabilises gradient flow in a deep CNN.
4. Demonstrate the train-vs-eval mode bug and how to avoid it.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Batch Normalization Forward Pass (NumPy)

In [ ]:
# BN normalises each feature across the batch then applies learnable scale/shift.

class BatchNormNumpy:
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        self.gamma = np.ones(num_features)
        self.beta  = np.zeros(num_features)
        self.eps   = eps
        self.momentum = momentum
        self.running_mean = np.zeros(num_features)
        self.running_var  = np.ones(num_features)

    def forward(self, x, training=True):
        if training:
            mean = x.mean(axis=0)
            var  = x.var(axis=0)
            # Update running stats for later inference use
            self.running_mean = (1 - self.momentum)*self.running_mean + self.momentum*mean
            self.running_var  = (1 - self.momentum)*self.running_var  + self.momentum*var
        else:
            mean, var = self.running_mean, self.running_var
        x_norm = (x - mean) / np.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

# Verify: output should be approx zero-mean, unit-variance per feature
bn = BatchNormNumpy(num_features=4)
X_in = np.random.randn(32, 4) * 5 + 3   # non-zero mean, large scale
X_out = bn.forward(X_in, training=True)
print(f"Input  -- mean: {X_in.mean(axis=0).round(3)},  std: {X_in.std(axis=0).round(3)}")
print(f"Output -- mean: {X_out.mean(axis=0).round(3)},  std: {X_out.std(axis=0).round(3)}")

# Inference mode should use running stats
X_infer = np.random.randn(10, 4) * 5 + 3
X_infer_out = bn.forward(X_infer, training=False)
print(f"Inference output mean (uses running_mean): {X_infer_out.mean(axis=0).round(3)}")
print("Batch-norm forward pass verified.")


## Level 2: BatchNorm vs LayerNorm vs No Norm (PyTorch)

In [ ]:
# Compare three normalisation strategies on a regression task.

def make_deep_net(norm_type, n_in=10):
    layers = []
    sizes = [n_in, 32, 64, 64, 32]
    for i in range(len(sizes)-1):
        in_f, out_f = sizes[i], sizes[i+1]
        layers.append(nn.Linear(in_f, out_f))
        if norm_type == "batch":
            layers.append(nn.BatchNorm1d(out_f))
        elif norm_type == "layer":
            layers.append(nn.LayerNorm(out_f))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(32, 1))
    return nn.Sequential(*layers).to(device)

N = 1000
Xd = torch.randn(N, 10)
yd = (Xd[:, 0]**2 + Xd[:, 1]*Xd[:, 2] + 0.1*torch.randn(N)).unsqueeze(1)
tr_ds = TensorDataset(Xd[:800].to(device), yd[:800].to(device))
va_ds = TensorDataset(Xd[800:].to(device), yd[800:].to(device))
tr_ld = DataLoader(tr_ds, batch_size=64, shuffle=True)
va_ld = DataLoader(va_ds, batch_size=64)

norm_results = {}
for ntype in ["batch", "layer", "none"]:
    model = make_deep_net(ntype)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    for epoch in range(100):
        model.train()
        for xb, yb in tr_ld:
            opt.zero_grad()
            try:
                crit(model(xb), yb).backward()
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    print(f"OOM with {ntype} norm")
                    torch.cuda.empty_cache()
                    continue
                raise
            opt.step()
    model.eval()
    with torch.no_grad():
        vl = sum(crit(model(xb), yb).item() * len(xb) for xb, yb in va_ld) / len(va_ld.dataset)
    norm_results[ntype] = vl
    print(f"Norm={ntype:5s}  val MSE={vl:.5f}")

best_norm = min(norm_results, key=norm_results.get)
print(f"Best normalisation strategy: {best_norm}")


## Real-World Example 1: BN Stabilises Gradient Flow in a Deep CNN

In [ ]:
# Measure gradient norms per layer with and without BatchNorm.

def build_cnn(use_bn, depth=6):
    layers = [nn.Conv2d(1, 16, 3, padding=1)]
    if use_bn:
        layers.append(nn.BatchNorm2d(16))
    layers.append(nn.ReLU())
    for _ in range(depth - 1):
        layers += [nn.Conv2d(16, 16, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(16))
        layers += [nn.ReLU()]
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)]
    return nn.Sequential(*layers).to(device)

def measure_grad_norms(model, x, y):
    model.train()
    out = model(x)
    nn.CrossEntropyLoss()(out, y).backward()
    norms = [(name, p.grad.norm().item())
             for name, p in model.named_parameters() if p.grad is not None]
    return norms

x_img = torch.randn(32, 1, 16, 16, device=device)
y_cls = torch.randint(0, 10, (32,), device=device)

for use_bn in [False, True]:
    cnn = build_cnn(use_bn)
    gnorms = [v for _, v in measure_grad_norms(cnn, x_img, y_cls)]
    tag = "With BN   " if use_bn else "Without BN"
    ratio = max(gnorms) / max(min(gnorms), 1e-12)
    print(f"{tag}: min_grad={min(gnorms):.2e}  max_grad={max(gnorms):.2e}  ratio={ratio:.1f}x")

print("With BN: gradient ratio is much smaller -- no vanishing/exploding gradients.")


## Real-World Example 2: BatchNorm Train vs Eval Mode Bug

In [ ]:
# Classic production bug: forgetting model.eval() before inference.
# In train mode, BN uses batch statistics -- predictions change with batch composition.

class TinyBNNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Linear(16, 2)
        )
    def forward(self, x):
        return self.net(x)

X_bug = torch.randn(100, 4).to(device)
y_bug = torch.randint(0, 2, (100,)).to(device)
model_bug = TinyBNNet().to(device)
opt_bug = torch.optim.Adam(model_bug.parameters(), lr=1e-2)
crit_bug = nn.CrossEntropyLoss()

for _ in range(30):
    model_bug.train()
    opt_bug.zero_grad()
    crit_bug(model_bug(X_bug), y_bug).backward()
    opt_bug.step()

x_test = X_bug[:1]
with torch.no_grad():
    model_bug.train()          # BUG: should call model.eval() first
    pred_train = model_bug(x_test).argmax(dim=-1).item()
    model_bug.eval()           # FIX
    pred_eval  = model_bug(x_test).argmax(dim=-1).item()

print(f"Prediction in TRAIN mode: {pred_train}")
print(f"Prediction in EVAL  mode: {pred_eval}")
if pred_train != pred_eval:
    print("BUG REPRODUCED: train-mode BN gives wrong prediction due to batch statistics!")
else:
    print("Predictions match (train more to make the difference visible).")
print("Rule: ALWAYS call model.eval() before inference, model.train() before training.")


## Real-World Example 3: Gradient Norms With vs Without BN (10-Layer MLP)

In [ ]:
# Quantify gradient vanishing in a deep MLP. Tanh is prone to vanishing --
# BN rescues gradient flow even in very deep networks.

def build_deep_mlp(use_bn, n_layers=10):
    layers = []
    in_f = 20
    for _ in range(n_layers):
        layers.append(nn.Linear(in_f, 32))
        if use_bn:
            layers.append(nn.BatchNorm1d(32))
        layers.append(nn.Tanh())   # tanh saturates -- worst case for vanishing
        in_f = 32
    layers.append(nn.Linear(32, 1))
    return nn.Sequential(*layers).to(device)

X_deep = torch.randn(64, 20, device=device)
y_deep = torch.randn(64, 1, device=device)

for use_bn in [False, True]:
    mdl = build_deep_mlp(use_bn)
    mdl.train()
    nn.MSELoss()(mdl(X_deep), y_deep).backward()
    grad_norms = [p.grad.norm().item()
                  for name, p in mdl.named_parameters()
                  if "weight" in name and p.grad is not None]
    tag = "With BN   " if use_bn else "Without BN"
    ratio = max(grad_norms) / max(min(grad_norms), 1e-12)
    print(f"{tag}: norms={[round(g,5) for g in grad_norms]}")
    print(f"           Max/min ratio: {ratio:.1f}x")

print("BN dramatically reduces gradient norm ratio in deep networks.")
